In [1]:
import cv2
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
import tensorflow
from tensorflow.keras.models import load_model

In [5]:
# carregando o modelo
path = '../Material/Material/'
model = load_model(path + "modelo_02_expressoes.h5")

In [7]:
# carregando o vídeo
archive_video = path + "Videos/video_teste06.MOV"
cap = cv2.VideoCapture(archive_video)

conected, video = cap.read()
print(video.shape)

(720, 1276, 3)


In [8]:
# define se vai redimensionar
resize = True
# se for redimennsionar, qual será a largura máxima
max_width = 600

# fazendo (ou nao) o redimensionamento
if (resize and video.shape[1]>max_width):  
  proportion = video.shape[1] / video.shape[0]
  video_width = max_width
  video_height = int(video_width / proportion)
else:
  video_width = video.shape[1]
  video_height = video.shape[0]

In [10]:
# nome do arquivo de vídeo que será salvo
name_archive = 'resultado_video_teste06.avi'

# definição do codec
fourcc = cv2.VideoWriter_fourcc(*'XVID')
fps = 24

# saida do vídeo
video_output = cv2.VideoWriter(name_archive, fourcc, fps, (video_width, video_height))

In [11]:
from tensorflow.keras.preprocessing.image import img_to_array

In [12]:
# define se vai ficar no modo detalhado (só funciona para uma face)
unique_face = True

# cascade para reconher a face
haarcascade_faces = path + 'haarcascade_frontalface_alt.xml'

# define os tamanhos para as fontes e o tipo
little_font, medium_font = 0.4, 0.7
font = cv2.FONT_HERSHEY_SIMPLEX

# expressoes faciais
expressions = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]

# loop para detecção no vídeo
while (cv2.waitKey(1) < 0):
    conected, frame = cap.read()

    # se não conectou já encerra
    if not conected:
        break 

    # tempo
    t = time.time()

    # redimensiona o frame do vídeo
    if resize:
      frame = cv2.resize(frame, (video_width, video_height))

    # carrega o cascade para reconhecer as faces
    face_cascade = cv2.CascadeClassifier(haarcascade_faces)
    # converte para escala de cinza
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # detecta as faces
    faces = face_cascade.detectMultiScale(gray,scaleFactor=1.2, minNeighbors=5,minSize=(30,30))

    # se detectou faces
    if len(faces) > 0:
        for (x, y, w, h) in faces:

            # se detectar mais de uma face então considera aquela que possui uma maior area na imagem
            if unique_face and len(faces) > 1:
                max_area_face = faces[0]
                for face in faces:
                    if face[2] * face[3] > max_area_face[2] * max_area_face[3]:
                        max_area_face = face
                face = max_area_face

                # coordenadas da maior face
                (x,y,w,h) = max_area_face

            # desenha retângulo ao redor da face
            frame = cv2.rectangle(frame,(x,y),(x+w,y+h+10),(255,50,50),2)

            # extrai apenas a região de interesse
            roi = gray[y:y + h, x:x + w]      
            # redimensiona
            roi = cv2.resize(roi, (48, 48))
            # normalização
            roi = roi.astype("float") / 255.0
            # converção para array
            roi = img_to_array(roi)
            # muda o formato para incluir a dimensão das cores (1 dimensão, grayscale)
            roi = np.expand_dims(roi, axis=0)

            # faz a predição
            result = model.predict(roi)[0]
            print(result)
            if result is not None:
                if unique_face:
                    for (index, (emotion, prob)) in enumerate(zip(expressions, result)):
                        # nomes das emoções
                        text = "{}: {:.2f}%".format(emotion, prob * 100)
                        # calcula do tamanho da barra, com base na probabilidade
                        bar = int(prob * 150)
                        left_space = 7
                        if bar <= left_space:
                            bar = left_space + 1
                        
                        # desenha na imagem
                        cv2.rectangle(frame, (left_space, (index * 18) + 7), (bar, (index * 18) + 18), (200, 250, 20), -1)
                        cv2.putText(frame, text, (15, (index * 18) + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.25, (0, 0, 0), 1, cv2.LINE_AA)

                # encontra a emoção com maior probabilidade
                result_new = np.argmax(result)

                # escreve a emoção acima do rosto
                cv2.putText(frame,expressions[result_new],(x,y-10), font, medium_font,(255,255,255),1,cv2.LINE_AA)

            # se tem apenas uma face, e já foi encontrada, então para o loop
            if unique_face and len(faces) > 1:
                break

    # tempo processado
    cv2.putText(frame, " frame processado em {:.2f} segundos".format(time.time() - t), (20, video_height-20), font, little_font, (250, 250, 250), 0, lineType=cv2.LINE_AA)

    cv2.imshow(frame)
    # grava o frame atual
    video_output.write(frame)

print("Terminou")
video_output.release()
cv2.destroyAllWindows()

[ERROR:0@1317.323] global persistence.cpp:566 open Can't open file: '../Material/Material/haarcascade_frontalface_alt.xml' in read mode


error: OpenCV(4.12.0) /home/task_176181477337787/conda-bld/opencv-suite_1761814819046/work/modules/objdetect/src/cascadedetect.cpp:1689: error: (-215:Assertion failed) !empty() in function 'detectMultiScale'
